In [ ]:
!pip install openai ipywidgets --quiet

In [ ]:
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, Markdown
import os

**API Key Input**

In [ ]:
os.environ["OPENAI_API_KEY"] = input("🔑 Enter your OpenAI API key: ")

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

**Transcript / Field Note Input**

Make sure to run the cell below and input your transcript/field note via either .txt file or just text!

In [ ]:
upload = widgets.FileUpload(accept='.txt', multiple=False)
transcript_box = widgets.Textarea(
    placeholder='Or paste transcript text here...',
    description='Transcript:',
    layout=widgets.Layout(width='100%', height='300px')
)

display(upload, transcript_box)

transcript = ""

def handle_upload(change):
    global transcript
    transcript = next(iter(upload.value.values()))['content'].decode('utf-8')
    transcript_box.value = transcript

upload.observe(handle_upload, names='value')

In [ ]:
transcript_box.value

**GPT Helper Function**

In [ ]:
def run_gpt(prompt):
    response = client.chat.completions.create(
        model="gpt-5.1-chat-latest",
        messages=[
            {"role": "system", "content": "You are a helpful research assistant analyzing interview transcripts for design insights."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content

## Prompts

Below are the prompts that will be used for the transcript analyzer. Please feel free to modify the prompts if you feel it is warranted as long as it preserves the original goal of the AI modules.

In [1]:
def keyword_extraction(text):
    return run_gpt(f"Extract 10–15 key terms and ideas that capture this transcript / field note:\n{text}")

def theme_detection(text):
    return run_gpt(f"Identify the main themes across this transcript / field note data and group related ideas. Provide short explanations for each theme:\n{text}")

def sentiment_check(text):
    return run_gpt(f"Summarize the emotional tone of this transcript /field note: positive, negative, neutral, mixed. Explain key quotes that illustrate emotion:\n{text}")

def summary_gen(text):
    return run_gpt(f"Write a concise, 150-word summary of the main points and design-relevant insights from this transcript / field note:\n{text}")

**Transcript / Field Note Analyzer**

Run the cells here and use one of the four tools available. For the custom prompt, you have more flexibility on how you wish to ask the AI to analyze the transcript / field note.

In [ ]:
analysis_types = ['Keyword Extraction', 'Theme Identification', 'Sentiment Analysis', 'Summary', 'Custom Prompt']
dropdown = widgets.Dropdown(options=analysis_types, description='Analysis:')
custom_prompt_box = widgets.Textarea(
    placeholder='Enter your custom question or instruction (used only when Custom Prompt selected)',
    layout=widgets.Layout(width='100%', height='100px')
)
run_button = widgets.Button(description='Run', button_style='success')
output_box = widgets.Output()

def on_click(b):
    output_box.clear_output()
    text = transcript_box.value
    task = dropdown.value
    with output_box:
        if not text.strip():
            display(Markdown("*Please upload or paste a transcript first.*"))
            return
        if task == 'Keyword Extraction':
            display(Markdown("### Extracted Keywords"))
            print(keyword_extraction(text))
        elif task == 'Theme Identification':
            display(Markdown("### Themes"))
            print(theme_detection(text))
        elif task == 'Sentiment Analysis':
            display(Markdown("### Sentiment"))
            print(sentiment_check(text))
        elif task == 'Summary':
            display(Markdown("### Summary"))
            print(summary_gen(text))
        elif task == 'Custom Prompt':
            prompt = custom_prompt_box.value or "Summarize this transcript."
            display(Markdown(f"### Response to custom prompt:\n> {prompt}"))
            full_prompt = f"Refer to this transcript:\n\n{text}\n\nNow: {prompt}"
            print(run_gpt(full_prompt))

run_button.on_click(on_click)
display(dropdown, custom_prompt_box, run_button, output_box)